In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# after47 固定单点数值诊断
复用 run08 的配置与 OFF1 浮点 RGB，仅重建 step0..47。一次梯度基线、两次无梯度相同基线，以及既定更新与反向扰动各一次。不会选择候选或修改 run08 结果，不生成 MP4。


In [ ]:
from pathlib import Path
import sys, subprocess, json
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_COMMIT = '891988bfeaedb82a622d9ae3aac7a3e1531d66d5'
SOURCE = Path('/content/public_statistic_phase2_source')
if SOURCE.exists(): raise FileExistsError('Use a fresh runtime; preserve existing source')
# Fetch the exact published source used before the branch rename.
subprocess.run(['git','init',str(SOURCE)],check=True)
subprocess.run(['git','-C',str(SOURCE),'remote','add','origin',REPOSITORY_URL],check=True)
subprocess.run(['git','-C',str(SOURCE),'fetch','--depth','1','origin',SOURCE_COMMIT],check=True)
subprocess.run(['git','-C',str(SOURCE),'checkout','--detach','FETCH_HEAD'],check=True)
actual_commit = subprocess.check_output(['git','-C',str(SOURCE),'rev-parse','HEAD'],text=True).strip()
assert actual_commit == SOURCE_COMMIT, (actual_commit, SOURCE_COMMIT)
print('Source:', actual_commit)


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','diffusers','transformers','accelerate','ftfy','sentencepiece','safetensors','huggingface_hub','numpy','Pillow'],check=True)
subprocess.run(['ffmpeg','-version'],check=True)
# Keep Colab CUDA PyTorch. Actual versions are recorded by the worker.


## 固定范围与预算
116 次 Transformer、5 次 VAE、1 次 backward；另计最多 120 次 block / 13 次 chunk 重算。0 MP4，1800 秒墙钟，零自动重试。显存和主机内存只记录，无显卡型号白名单或人工显存配额；计算路径需要 CUDA/BF16。
比较实际增量、g·Δz、基线重复波动及正负 loss；有限步长曲率和混合精度均可能导致差异，诊断候选不作为方法通过结果。只完成了 CPU 测试，真实 GPU 诊断尚未执行。


In [ ]:
BASE=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2/run08')
CONFIG=BASE/'config.json'
REFERENCE=BASE/'OFF1/terminal_float_rgb.npy'
OUTPUT=Path('/content/drive/MyDrive/Video-WM/public-statistic-phase2/after47-diagnostic01')
for path in (CONFIG,REFERENCE):
    if not path.is_file(): raise FileNotFoundError(path)
if OUTPUT.exists(): raise FileExistsError(str(OUTPUT))
print('Source run08 config:', CONFIG.read_text())
print('Diagnostic budget: 116 transformer / 5 VAE / 1 backward; 120 block / 13 chunk replay; 0 MP4; 1800 seconds')


In [ ]:
import os, signal
command=[sys.executable,'-m','experiments.public_statistic.run_after47_diagnostic','--config',str(CONFIG),'--reference',str(REFERENCE),'--output',str(OUTPUT)]
process=subprocess.Popen(command,cwd=SOURCE,start_new_session=True)
try:
    returncode=process.wait()
except BaseException:
    # Cancel the launcher; it forwards cancellation and preserves worker results.
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid,signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit',returncode)
print((OUTPUT/'result.json').read_text() if (OUTPUT/'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode,command)


## 回传
回传 after47-diagnostic01 下 result.json、execution_exit.json、loaded_model.json、runtime.json、recomputation.json 和 execution.log。after47_state_and_gradient.pt 保存固定点、梯度、实际正负增量、target 和 scheduler 状态，供进一步离线核验；它是本项目生成的 Python 对象快照。失败时保留已完成评估，不自动重试。
